# AI-Based Marketing Campaign Generator

## Problem Statement

### Business Context
Marketing teams spend significant time brainstorming a catchy **Title**, a compelling **email draft** and a strong **call-to-action (CTA)**. These directly influence engagement and conversions, but producing them consistently across many products is slow. AI-powered generation accelerates ideation, keeps messaging consistent and tailors content to each product's audience.

### Objective
Build an **AI-based Marketing Campaign Generator** that:
1. **Fetches product details** from a SQLite database by **Product ID** (name, target audience, specifics) using a **SQL agent**.
2. **Generates campaign assets** with an LLM + agent: **Marketing Title**, **Email Draft**, **CTA**.
3. **Outputs** coherent, personalised, safe and professional campaign-ready content.

### Data Description
| Field | Meaning | Example |
|---|---|---|
| `product_id` | Unique product code | P1011 |
| `product_name` | Name of the product | Golden Future Mutual Fund |
| `target_age` | Intended age range | 30-50 |
| `target_profession` | Intended profession | Salaried Professional |
| `product_details` | Features, benefits, strategy | ... |

### Architecture
```
Product ID ──► normalise ("1011" → "P1011")
          ──► SQL Agent (ReAct: list tables → schema → SELECT)  ──► product details text
          ──► Campaign Agent (ReAct) ──► Tool "Campaign Generator" (LLM + prompt)
          ──► Title + Email Draft + CTA   (optional: Pydantic structured output, Llama Guard check)
```

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Install LangChain + Groq + related integrations (pinned for reproducibility)
%pip install -qU \
    langchain==0.3.27 \
    langchain-community==0.3.27 \
    langchainhub==0.1.21 \
    langchain-experimental==0.3.4 \
    langchain-huggingface==0.3.1 \
    langchain-groq==0.3.8 \
    ddgs==9.6.1 \
    numpy==2.0.2

> **Note:** After running the above cell, restart the runtime (Colab) / kernel (Jupyter) and run all cells sequentially from the next cell. Dependency-resolver warnings can be ignored.

In [ ]:
import json
import os
import re
import sqlite3
import warnings

import pandas as pd
from langchain.agents import AgentType, Tool, initialize_agent
from langchain_community.agent_toolkits import SQLDatabaseToolkit, create_sql_agent
from langchain_community.utilities import SQLDatabase
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional, Dict

warnings.filterwarnings("ignore", category=DeprecationWarning)

## Creating the Groq API Key
1. Log in at <https://console.groq.com/login> → **API Keys** → **Create API Key** → copy it.
2. In Colab open the 🔑 **Secrets** panel, add `GROQ_API_KEY`, paste the key and toggle *Notebook access* on.

Outside Colab, set the environment variable `GROQ_API_KEY` or type it when prompted.

In [ ]:
try:
    from google.colab import userdata
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")   # read the key from Colab Secrets
except ImportError:
    if not os.getenv("GROQ_API_KEY"):
        from getpass import getpass
        os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY: ")

### Initialise the Groq models with different temperature settings
*Temperature* rescales the next-token probability distribution. `0` ≈ deterministic (best for SQL / facts); higher values = more diverse wording. **Groq accepts 0–2.**

In [ ]:
from langchain_groq import ChatGroq   # Groq LLM interface

groq_api_key = os.environ["GROQ_API_KEY"]
MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

# High-creativity LLM (max temperature on Groq is 2)
llm_high = ChatGroq(model=MODEL, temperature=2, groq_api_key=groq_api_key)

# Low-creativity (deterministic) LLM -- used for SQL and the agents' reasoning
llm = ChatGroq(model=MODEL, temperature=0, groq_api_key=groq_api_key)

### Important: free-tier limits
Groq's free tier has per-minute / per-day limits. If you hit them: wait 12–24 h, or switch `MODEL` to another Groq model listed in the Groq playground.

### Checking the LLMs' responses

In [ ]:
# .invoke() replaces the deprecated .predict(); it returns an AIMessage
print(llm.invoke("Hello, how are you?").content)

In [ ]:
print(llm_high.invoke("Hello, how are you?").content)

## Create / Load the Product Database
The course provides `products.db` (upload it to `/content/`). The cell below recreates it if it is missing so the notebook is self-contained.

In [ ]:
DB_PATH = "/content/products.db" if os.path.isdir("/content") else "products.db"

PRODUCTS = [
    ("P1011", "Golden Future Mutual Fund", "30-50", "Salaried Professional",
     "The Golden Future Mutual Fund is designed for professionals looking to build long-term wealth through a blend of growth and stability. It invests in a diversified portfolio of large-cap stocks and emerging market equities, aiming to provide robust returns over time. Along with active portfolio management and periodic reviews, investors can expect a balanced approach to wealth creation."),
    ("P1012", "Golden Years Savings Account", "60-75", "Retired Professional",
     "Specifically tailored for retirees, the Golden Years Savings Account provides a secure and accessible way to manage post-retirement finances. With competitive interest rates, minimal fees, and hassle-free digital banking, this account ensures that your savings grow steadily with added benefits like loan facilities and investment options."),
    ("P1013", "Balanced Growth Investment Fund", "30-55", "Salaried Professional",
     "This fund offers a balanced investment strategy that combines the growth potential of equities with the relative safety of fixed-income securities. Ideal for mid-career professionals planning for future family needs and retirement, the Balanced Growth Investment Fund focuses on diversified asset allocation to optimize returns while managing risk."),
    ("P1014", "Senior Advantage Fixed Deposit", "60-75", "Retired and Semi-Retired Professionals",
     "The Senior Advantage Fixed Deposit is designed exclusively for retirees looking for secure and higher returns on their savings. Enjoy attractive interest bonuses, flexible tenure options, and minimal penalties for urgent withdrawals. Detailed risk assessments and regular reviews ensure that your investments are aligned with your financial goals."),
    ("P1015", "Dynamic Opportunity Savings & Investment Combo", "40-65", "Salaried and Semi-Retired Professionals",
     "This innovative product combines the benefits of a high-interest savings account with the growth potential of a low-to-moderate risk investment portfolio. Ideal for professionals transitioning into retirement or seeking a hybrid financial solution, the Dynamic Opportunity Combo offers flexible account management and the potential for higher returns on savings."),
]

if not os.path.exists(DB_PATH):
    with sqlite3.connect(DB_PATH) as conn:
        conn.execute("""CREATE TABLE products (
            product_id TEXT PRIMARY KEY, product_name TEXT NOT NULL, target_age TEXT,
            target_profession TEXT, product_details TEXT)""")
        conn.executemany("INSERT INTO products VALUES (?, ?, ?, ?, ?)", PRODUCTS)

with sqlite3.connect(DB_PATH) as conn:
    display(pd.read_sql("SELECT product_id, product_name, target_age, target_profession FROM products", conn))

## Build SQL Agent
The SQL agent is a **ReAct** agent whose tools come from `SQLDatabaseToolkit`: `sql_db_list_tables`, `sql_db_schema`, `sql_db_query_checker`, `sql_db_query`. The database is opened **read-only** so an LLM-written query can never modify data.

In [ ]:
# Load the SQLite database (read-only URI)
db = SQLDatabase.from_uri(f"sqlite:///file:{DB_PATH}?mode=ro&uri=true")
print(db.dialect, db.get_usable_table_names())

# System message for the SQL agent
system_message = "You are an SQL assistant. Your role is to efficiently fetch and retrieve product data from the database. Only run read-only SELECT queries."

# Toolkit = database + LLM (the LLM is used by the query-checker tool)
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

In [ ]:
# Initialise the SQL agent
sqlite_agent = create_sql_agent(
    llm=llm,                                   # the "brain"
    toolkit=toolkit,                           # tools + database
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,   # "zero-shot-react-description"
    prefix=system_message + "\n\nYou have access to the following tools:",
    verbose=False,
    max_iterations=8,
    agent_executor_kwargs={"handle_parsing_errors": True},
)

In [ ]:
query = "Fetch all the unique product_id with product_name"
output = sqlite_agent.invoke(query)
output

In [ ]:
query = "Fetch all the products details with product_id"
output = sqlite_agent.invoke(query)
output

## Build Marketing Campaign Agent
### Campaign Generation Function
A plain Python function wrapped as a LangChain **Tool**. The agent reads the tool's `description` to decide when to call it.

In [ ]:
CAMPAIGN_PROMPT = """
You are a marketing campaign generator. Your task is to create a compelling marketing title, an engaging email draft, and a strong call-to-action (CTA) based on the provided product details. Ensure the content is professional, polite, and aligned with the product's marketing objectives. Do not invent returns, guarantees or numbers that are not in the product details.

### Product Details:
{product_details}

Please generate:
1. **Marketing Title:** (A short, catchy, and memorable tagline)
2. **Email Draft:** (A professional, polite, and engaging email to promote the product, around 100-150 words)
3. **Call-to-Action (CTA):** (A persuasive instruction that motivates the target audience to take the desired action)
"""

def campaign_generation_func(input: str) -> str:
    prompt = CAMPAIGN_PROMPT.format(product_details=input)
    return llm.invoke([HumanMessage(content=prompt)]).content.strip()

### Initialize Marketing Campaign Agent

In [ ]:
tools = [
    Tool(
        name="Campaign Generator",
        func=campaign_generation_func,
        description="Generates marketing campaign assets (title, email, CTA) based on product details. Input should be the full product details text.",
    )
]

agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=False,
    handle_parsing_errors=True,
    max_iterations=4,
)

## Build a Marketing Campaign Generator

In [ ]:
def normalize_product_id(raw: str) -> str:
    """'1011', 'p1011', ' P1011 ' -> 'P1011'. Rejects anything else (also blocks prompt/SQL injection)."""
    m = re.fullmatch(r"\s*[pP]?\s*(\d{3,6})\s*", raw or "")
    if not m:
        raise ValueError(f"Invalid Product ID: {raw!r}")
    return f"P{m.group(1)}"


def campaign_generator(product_id: Optional[str] = None):
    p_id = normalize_product_id(product_id or input("Enter Product ID (e.g., P1011): "))

    # 1) Fetch product details with the SQL agent
    product_details = sqlite_agent.invoke(f"Fetch all columns for Product ID {p_id}")
    product_detail = product_details["output"]

    # 2) Prompt for the campaign agent
    agent_prompt = f"Generate a marketing campaign for the following product details: {product_detail}"

    # 3) Generate campaign content
    response = agent.invoke({"input": agent_prompt})["output"]
    print(response)
    return response

## Test Product IDs
For each Product ID: run the cell, type the ID in the input box (e.g. `P1011` or just `1011`), press **Enter**.

### Product ID: P1011

In [ ]:
campaign_generator()

**Observations:** _write what you notice about tone, audience fit and CTA strength here._

### Product ID: P1012

In [ ]:
campaign_generator()

**Observations:** _write what you notice about tone, audience fit and CTA strength here._

### Product ID: P1013

In [ ]:
campaign_generator()

**Observations:** _write what you notice about tone, audience fit and CTA strength here._

### Product ID: P1014

In [ ]:
campaign_generator()

**Observations:** _write what you notice about tone, audience fit and CTA strength here._

### Product ID: P1015

In [ ]:
campaign_generator()

**Observations:** _write what you notice about tone, audience fit and CTA strength here._

## Bonus 1 – Structured Output with Pydantic
Free text is hard to plug into an email tool. `with_structured_output` makes the model return JSON that is validated against a Pydantic schema.

In [ ]:
class CampaignAssets(BaseModel):
    title: str = Field(description="Short, catchy, memorable marketing tagline")
    email_subject: str = Field(description="Subject line of the email")
    email_body: str = Field(description="Professional, polite, engaging email of 100-150 words")
    cta: str = Field(description="Persuasive call-to-action")

structured_llm = llm.with_structured_output(CampaignAssets)

with sqlite3.connect(DB_PATH) as conn:     # deterministic, parameterised lookup -- no agent needed
    row = pd.read_sql("SELECT * FROM products WHERE product_id = ?", conn, params=("P1013",)).iloc[0]

assets = structured_llm.invoke(CAMPAIGN_PROMPT.format(product_details=row.to_string()))
print(json.dumps(assets.model_dump(), indent=2))

## Bonus 2 – A/B Variants with the High-Temperature Model
Generate several Title/CTA options for rapid A/B testing. (Temperature 2 can produce gibberish; ~1.0 is a safer creative setting.)

In [ ]:
creative = ChatGroq(model=MODEL, temperature=1.0, groq_api_key=groq_api_key).with_structured_output(CampaignAssets)
for i in range(3):
    v = creative.invoke(CAMPAIGN_PROMPT.format(product_details=row.to_string()))
    print(f"Variant {i+1}: {v.title}  |  CTA: {v.cta}")

## Bonus 3 – Safety Check with Llama Guard
The objective asks for *safe, professional outputs*. A guard model classifies text as `safe` or `unsafe` + category codes.

In [ ]:
guard = ChatGroq(model="meta-llama/llama-guard-4-12b", temperature=0, groq_api_key=groq_api_key)

def is_safe(text: str) -> bool:
    return guard.invoke([HumanMessage(content=text)]).content.strip().lower().startswith("safe")

print("Campaign is safe:", is_safe(assets.email_body))

## Conclusion

### Actionable Insights and Recommendations
- **Scalability & efficiency:** drafts for hundreds of products in minutes without growing headcount.
- **Consistency & accuracy:** product facts come straight from the database, so copy matches the product and its target audience.
- **Human-in-the-loop:** keep a reviewer for brand voice, compliance (financial products!) and final polish.
- **A/B testing:** use a higher-temperature model to produce multiple Titles/CTAs and measure open/click rates.
- **Guardrails:** read-only DB, validated Product IDs, structured output, and a Llama Guard check before anything is sent.

### Key Takeaways
- Agentic AI (SQL agent + tool-using campaign agent) personalises campaigns per product and audience.
- Fetching data from a verified database reduces hallucinated product claims.
- Automating Title/Email/CTA reduces creative fatigue and time-to-market.